# Helper notebook for results administration

This notebook helps manage the repository `results/` folder.

It can solve two practical tasks:

1. **Build a parameter-set overview table** to see what parameter set hides behind the hashed result folders names.
2. **Delete outdated result folders safely** to clean `results/` repo when old and new runs become mixed.

Quick `runner.py` logic explanation:

- results are stored under `results/<ModelName>_sweep/params_<hash>/...` or `results/<ModelName>_timecourse/params_<hash>/...`
- the hash belongs to the **effective parameter set**, not to a single run
- each `params_<hash>` folder may contain a `param_set.json` manifest
- each run folder may contain `run_config.json`, `run_summary.json`, and `run.h5`


## How to use this notebook

Run the notebook from top to bottom.

- First, choose which models and result kinds to scan.
- Then run the scan cell to build the Param-set table.
- Use the deletion helpers only when you really want to clean old folders.

Deletion is intentionally conservative:

- preview mode (`dry_run=True`) is the default
- deletion is only allowed inside the repository `results/` folder
- destructive deletion requires `confirm="DELETE"`


In [12]:
# Imports and repository setup

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts import runner

print(f"repo_root: {repo_root}")

repo_root: C:\Users\LevinSchöpfer\Automation_main


## User input

This is the only section that normally needs editing.

### `MODELS`
Use either:

- `"all"` to scan every detected model under `results/`, or
- a list such as `["Walz", "Kapitanov"]`

### `KINDS`
Choose which result trees to scan:

- `"sweep"`
- `"timecourse"`

### Display flags
These only affect what is shown in the notebook. They do not change the scan itself.


In [ ]:
# -------------------------
# INPUT — edit only here
# -------------------------

MODELS = "all"                    # "all" or list like ["Walz", "Kapitanov"]
KINDS = ["sweep", "timecourse"] # any subset of ["sweep", "timecourse"]

SHOW_PARAM_TABLE = True
SHOW_MANIFEST_ERRORS = True
SHOW_EFFECTIVE_PARAMS_IN_TABLE = False

# Number of rows displayed directly in the notebook.
# The full DataFrame remains available as df_param_sets.
DISPLAY_MAX_ROWS = 200

## Build the Param-set table

This cell contains both the helper functions and the calling code.

The table has **one row per `params_<hash>` folder** and summarizes:

- model
- result kind (`sweep` or `timecourse`)
- parameter hash
- creation timestamp from `param_set.json`
- number of run folders found below that parameter set
- dose values simulated
- interval values simulated
- `n_doses` values simulated
- parameter overrides that define the parameter set

This table is intentionally the only summary table in the notebook. The older run-level table was removed because it added a lot of processing time without adding much practical value for folder administration.


In [ ]:
# -------------------------
# INPUT (edit only here)
# -------------------------
RESULTS_DIR = Path("results")

# Choose models:
# - "all" to scan all detected models
# - or a list like ["Lebri", "Walz"]
MODELS = ["Lebri"]

# Choose result kinds:
# - any subset of ["sweep", "timecourse"]
KINDS = ["sweep", "timecourse"]

# Show full text columns without truncation
EXPAND_DISPLAY = False

# Maximum number of rows to display
MAX_ROWS = 200
# -------------------------


def detect_models(results_dir: Path):
    """Detect model names from folders such as Walz_sweep or Walz_timecourse."""
    if not results_dir.exists():
        return []

    models = set()
    for p in results_dir.iterdir():
        if not p.is_dir():
            continue
        name = p.name
        if name.endswith("_sweep"):
            models.add(name[:-6])
        elif name.endswith("_timecourse"):
            models.add(name[:-11])
    return sorted(models)


def normalize_models(results_dir: Path, models):
    """Resolve 'all' into all detected model names."""
    detected = detect_models(results_dir)
    if models == "all":
        return detected
    if isinstance(models, str):
        if models.lower() == "all":
            return detected
        return [models]
    return list(models)


def safe_read_json(path: Path):
    """Read a JSON file safely. Return None on failure."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def compact_dict(d):
    """Convert a dict into a short readable string for table display."""
    if not isinstance(d, dict) or not d:
        return ""
    return ", ".join(f"{k}={v}" for k, v in sorted(d.items()))


def summarize_values(values):
    """Return a sorted compact list for stable display."""
    clean = []
    for v in values:
        if v is None:
            continue
        if v not in clean:
            clean.append(v)
    try:
        return sorted(clean)
    except Exception:
        return clean


def collect_param_set_row(param_root: Path, model: str, kind: str):
    """
    Build one summary row for one params_<hash> folder.
    Uses:
      - param_set.json for parameter metadata
      - run_config.json / run_summary.json / run.h5 presence to summarize runs
    """
    manifest_path = param_root / "param_set.json"
    manifest = safe_read_json(manifest_path)

    manifest_error = manifest is None

    param_overrides = {}
    created_at = None

    if manifest:
        param_overrides = manifest.get("param_overrides", {}) or {}
        created_at = manifest.get("created_at", None)

    run_dirs = [p for p in param_root.iterdir() if p.is_dir()]

    dose_values = []
    interval_values = []
    n_doses_values = []
    n_runs_total = 0
    n_runs_with_summary = 0
    n_runs_with_config = 0
    n_runs_with_h5 = 0

    for run_dir in run_dirs:
        n_runs_total += 1

        summary_path = run_dir / "run_summary.json"
        config_path = run_dir / "run_config.json"
        h5_path = run_dir / "run.h5"

        if summary_path.exists():
            n_runs_with_summary += 1
        if config_path.exists():
            n_runs_with_config += 1
        if h5_path.exists():
            n_runs_with_h5 += 1

        cfg = safe_read_json(config_path)
        if cfg:
            run_params = cfg.get("run_params", {}) or {}
            dose_values.append(run_params.get("dose_mgkg"))
            interval_values.append(run_params.get("interval_weeks"))
            n_doses_values.append(run_params.get("n_doses"))

    return {
        "model": model,
        "kind": kind,
        "params_hash": param_root.name.replace("params_", "", 1),
        "created_at": created_at,
        "n_runs_total": n_runs_total,
        "n_runs_with_summary": n_runs_with_summary,
        "n_runs_with_config": n_runs_with_config,
        "n_runs_with_h5": n_runs_with_h5,
        "dose_values": summarize_values(dose_values),
        "interval_values": summarize_values(interval_values),
        "n_doses_values": summarize_values(n_doses_values),
        "param_overrides": compact_dict(param_overrides),
        "param_root": str(param_root),
        "manifest_error": manifest_error,
    }


def build_param_set_table(results_dir: Path, models, kinds):
    """Scan results folders and build one dataframe with one row per params_<hash> folder."""
    rows = []
    selected_models = normalize_models(results_dir, models)

    for model in selected_models:
        for kind in kinds:
            kind_root = results_dir / f"{model}_{kind}"
            if not kind_root.exists():
                continue

            for param_root in sorted(kind_root.glob("params_*")):
                if not param_root.is_dir():
                    continue
                rows.append(collect_param_set_row(param_root, model, kind))

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    sort_cols = [c for c in ["model", "kind", "created_at", "params_hash"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, na_position="last").reset_index(drop=True)

    return df


if EXPAND_DISPLAY:
    pd.set_option("display.max_colwidth", None)
else:
    pd.set_option("display.max_colwidth", 80)

pd.set_option("display.max_rows", MAX_ROWS)

detected_models = detect_models(RESULTS_DIR)
selected_models = normalize_models(RESULTS_DIR, MODELS)

print("Detected models under results/:", detected_models)
print("Selected models:", selected_models)
print("Selected kinds:", KINDS)

param_df = build_param_set_table(RESULTS_DIR, MODELS, KINDS)

print(f"Param-set rows: {len(param_df)}")
display(param_df)

## Safe deletion helpers

This section provides controlled deletion utilities.

### Safety rules

1. preview-only is the default: `dry_run=True`
2. destructive deletion requires `confirm="DELETE"`
3. only paths inside the repository `results/` folder are accepted

### Deletion levels

- `delete_param_set_folder(...)`: remove one whole `params_<hash>` folder
- `delete_model_kind_param_sets(...)`: remove all parameter sets inside one model/kind tree


In [ ]:
# Safe deletion helpers


def ensure_under_results(path: str | Path) -> Path:
    """Allow deletion only for paths located under repo_root/results."""
    p = Path(path).resolve()
    root = results_root().resolve()

    try:
        p.relative_to(root)
    except ValueError as exc:
        raise ValueError(f"Refusing to touch path outside results/: {p}") from exc

    return p


def _delete_path(path: str | Path, dry_run: bool = True, confirm: str | None = None):
    """Internal deletion helper used by the public delete functions."""
    p = ensure_under_results(path)

    if not p.exists():
        print(f"Path does not exist: {p}")
        return

    if dry_run:
        print(f"[DRY RUN] Would delete: {p}")
        return

    if confirm != "DELETE":
        raise ValueError('Refusing deletion. Set dry_run=False and confirm="DELETE" to proceed.')

    shutil.rmtree(p)
    print(f"Deleted: {p}")


def delete_param_set_folder(path: str | Path, dry_run: bool = True, confirm: str | None = None):
    """Delete one params_<hash> folder."""
    _delete_path(path, dry_run=dry_run, confirm=confirm)


def delete_model_kind_param_sets(model_name: str, kind: str, dry_run: bool = True, confirm: str | None = None):
    """Delete every params_<hash> folder under results/<Model>_<kind>."""
    root = _kind_root(model_name, kind)
    if not root.exists():
        print(f"No such root: {root}")
        return

    param_roots = sorted([p for p in root.iterdir() if p.is_dir() and p.name.startswith("params_")])
    if not param_roots:
        print(f"No params_<hash> folders found under: {root}")
        return

    for p in param_roots:
        _delete_path(p, dry_run=dry_run, confirm=confirm)

### Example deletion calls

These stay commented out so that nothing is removed accidentally.


In [ ]:
# Example 1: preview deletion of one parameter set
# delete_param_set_folder(
#     df_param_sets.loc[0, "param_root"],
#     dry_run=True,
# )

# Example 2: actually delete one parameter set after preview
# delete_param_set_folder(
#     df_param_sets.loc[0, "param_root"],
#     dry_run=False,
#     confirm="DELETE",
# )

# Example 3: preview deletion of all Walz timecourse parameter sets
# delete_model_kind_param_sets(
#     model_name="Walz",
#     kind="timecourse",
#     dry_run=True,
# )

## Notes on interpretation

A few details are worth keeping in mind when reading the Param-set table:

- the parameter hash belongs to the **effective parameter set**, not to a single dose or interval
- all run folders under the same `params_<hash>` therefore share the same base parameterization
- `dose_values`, `interval_values`, and `n_doses_values` are aggregated from the run folders below that parameter set
- `param_overrides` comes from `param_set.json` and describes the values that distinguish this hashed folder from other parameter sets
- if a manifest is missing or unreadable, the parameter set can still be listed, but some metadata may be incomplete
